In [11]:
from pathlib import Path
import pandas as pd
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from scipy.spatial.distance import pdist, squareform
from nilearn.image import resample_to_img

base_dir = Path(r"D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject")



##讀取特定subject的特定concepts trial index

def load_and_filter_conditions(
    base_dir,
    sub_id,
    targets,
    filename="subject_fMRI_nii/sub-01_condition_with_ratings.csv",
    verbose=True
):
    """
    Load + filter + sort fMRI condition table
    """

    # -------------------------
    # 1. load data
    # -------------------------
    path = base_dir / filename.replace("sub-01", sub_id)
    df = pd.read_csv(path)

    # -------------------------
    # 2. filter concepts
    # -------------------------
    df_selected = df[df["concept"].isin(targets)].copy()

    # -------------------------
    # 3. ordering
    # -------------------------
    df_selected["concept"] = pd.Categorical(
        df_selected["concept"],
        categories=targets,
        ordered=True
    )

    df_selected = (
        df_selected
        .sort_values(by=["concept"])
        .reset_index(drop=True)
    )

    # -------------------------
    # 4. print (optional)
    # -------------------------
    if verbose:
        print(f"\n[{sub_id}] total selected:", len(df_selected))
        print(df_selected["concept"].value_counts())
        print(df_selected)

    return df_selected



##由篩選過的trial index讀取部分concepts的fMRI trial

def extract_fmri_volumes_rowwise(
    df_sorted,
    base_dir,
    sub_id="sub-01",
    verbose=True
):
    """
    Extract fMRI volumes strictly following df row order.

    Returns
    -------
    selected_volumes : np.ndarray
        shape = (x, y, z, n_trials)
    """

    selected_volumes = []

    for row in df_sorted.itertuples():

        img_path = (
            base_dir /
            "subject_fMRI_nii" /
            sub_id /
            row.session /
            f"{sub_id}_{row.session}_run-{row.run:02d}_betas.nii"
        )

        img = nib.load(img_path)
        data = img.get_fdata()

        t = int(row.trial_idx)
        vol = data[..., t]

        selected_volumes.append(vol)

    selected_volumes = np.stack(selected_volumes, axis=-1)

    if verbose:
        print("Final shape:", selected_volumes.shape)
        print("mean:", np.mean(selected_volumes))
        print("std:", np.std(selected_volumes))
        print("min:", np.min(selected_volumes))
        print("max:", np.max(selected_volumes))

    return selected_volumes




##讀取並轉換atlas

reference_img = nib.load(base_dir / "subject_fMRI_nii\sub-01\ses-things01\sub-01_ses-things01_run-01_betas.nii")

atlas_img = nib.load(base_dir / "HCP_atlas\MNI_Glasser_HCP_v1.0.nii.gz")
atlas = nib.load(base_dir / "HCP_atlas\MNI_Glasser_HCP_v1.0.nii.gz").get_fdata()
print(atlas.shape)
print(atlas_img.affine)

##對atlas進行線性轉換，符合fMRI結構
print("beta shape:\n", reference_img.shape)
print("atlas shape:\n", atlas_img.shape)

print("beta affine:\n", reference_img.affine)
print("atlas affine:\n", atlas_img.affine)


atlas_rs = resample_to_img(atlas_img, reference_img, interpolation="nearest")
atlas = atlas_rs.get_fdata()

print("===resampled atlas info===")
print("resampled atlas shape:\n", atlas_rs.shape)
print("resampled atlas affine:\n", atlas_rs.affine)

##讀取atlas region名稱
tree = ET.parse(base_dir / "HCP_atlas\HCP-Multi-Modal-Parcellation-1.0.xml")
root = tree.getroot()

labels = {}

for label in root.find("data").findall("label"):
    idx = int(label.attrib["index"])
    name = label.text.strip()
    if name != "*.*.*.*.*":
        labels[idx] = name

print(labels)

df_labels = pd.DataFrame.from_dict(
    labels,
    orient="index",
    columns=["name"]
).reset_index().rename(columns={"index": "roi_id"})

def parse_name(name):
    try:
        hemi, region = name.split("_", 1)
        return hemi, region
    except:
        return None, name

df_labels[["hemi", "region"]] = df_labels["name"].apply(
    lambda x: pd.Series(parse_name(x))
)
print(df_labels.head())



##找尋特定roi id
def search_roi(df_labels, keyword):
    result = df_labels[
        df_labels["name"].str.contains(keyword, case=False, na=False)
    ].copy()

    return result.sort_values("roi_id")



##由已有的fMRI影像繪製特定roi的RDM

def compute_roi_rdm(
    atlas,
    selected_volumes,
    roi_ids,
    title="ROI RDM",
    metric="correlation",
    plot=True
):
    """
    atlas: 3D label map (x,y,z)
    selected_volumes: 4D (x,y,z,n_trials)
    roi_ids: list of ROI indices
    """

    # -------------------------
    # 1. ROI mask
    # -------------------------
    roi_mask = np.isin(atlas, roi_ids)

    print("ROI voxel count:", np.sum(roi_mask))

    # -------------------------
    # 2. extract voxel x trials
    # -------------------------
    roi_data = selected_volumes[roi_mask]

    print("ROI data shape (voxels, trials):", roi_data.shape)

    # safety check
    if roi_data.shape[0] == 0:
        raise ValueError("ROI mask is empty. Check roi_ids.")

    # -------------------------
    # 3. reshape for RSA
    # -------------------------
    trial_pattern = roi_data.T  # (trials, voxels)

    # -------------------------
    # 4. compute RDM
    # -------------------------
    brain_rdm = squareform(
        pdist(trial_pattern, metric=metric)
    )

    print("RDM shape:", brain_rdm.shape)

    # -------------------------
    # 5. plot
    # -------------------------
    if plot:
        fig, ax = plt.subplots(figsize=(8, 8))

        im = ax.imshow(brain_rdm, interpolation="none")

        ax.set_title(title)
        plt.colorbar(im, ax=ax, label="distance")

        plt.tight_layout()
        plt.show()

    return roi_mask, roi_data, brain_rdm

(256, 256, 256)
[[  -1.    0.    0.  128.]
 [   0.    1.    0. -146.]
 [   0.    0.    1. -108.]
 [   0.    0.    0.    1.]]
beta shape:
 (72, 91, 75, 92)
atlas shape:
 (256, 256, 256)
beta affine:
 [[  2.           0.           0.         -71.1632843 ]
 [  0.           2.           0.         -80.94287109]
 [  0.           0.           2.         -68.46676636]
 [  0.           0.           0.           1.        ]]
atlas affine:
 [[  -1.    0.    0.  128.]
 [   0.    1.    0. -146.]
 [   0.    0.    1. -108.]
 [   0.    0.    0.    1.]]
===resampled atlas info===
resampled atlas shape:
 (72, 91, 75)
resampled atlas affine:
 [[  2.           0.           0.         -71.1632843 ]
 [  0.           2.           0.         -80.94287109]
 [  0.           0.           2.         -68.46676636]
 [  0.           0.           0.           1.        ]]
{1: 'L_V1', 2: 'L_MST', 3: 'L_V6', 4: 'L_V2', 5: 'L_V3', 6: 'L_V4', 7: 'L_V8', 8: 'L_4', 9: 'L_3b', 10: 'L_FEF', 11: 'L_PEF', 12: 'L_55b', 13: 'L_

In [19]:
affordance_candidates_list = ['calculator', 'typewriter', 'pencil', 'spoon']
    

In [20]:
df_sub01 = load_and_filter_conditions(
    base_dir=base_dir,
    sub_id="sub-01",
    targets=affordance_candidates_list
)


[sub-01] total selected: 48
concept
calculator    12
typewriter    12
pencil        12
spoon         12
Name: count, dtype: int64
                   image_filename       session  run  trial_idx subject  \
0   calculator/calculator_09s.jpg  ses-things12   10         26  sub-01   
1   calculator/calculator_07s.jpg  ses-things07    1         87  sub-01   
2   calculator/calculator_11s.jpg  ses-things06    4          7  sub-01   
3   calculator/calculator_06s.jpg  ses-things05    3         76  sub-01   
4   calculator/calculator_03s.jpg  ses-things09    8         50  sub-01   
5   calculator/calculator_10n.jpg  ses-things04    4          9  sub-01   
6   calculator/calculator_01b.jpg  ses-things03    5         54  sub-01   
7   calculator/calculator_08s.jpg  ses-things10    2         73  sub-01   
8   calculator/calculator_02s.jpg  ses-things08    3          1  sub-01   
9   calculator/calculator_12s.jpg  ses-things01    9          3  sub-01   
10  calculator/calculator_05s.jpg  ses-thing

In [21]:
selected_volumes = extract_fmri_volumes_rowwise(
    df_sorted=df_sub01,
    base_dir=base_dir,
    sub_id="sub-01"
)

OSError: [WinError 1455] 分頁檔太小，無法完成操作。

In [ ]:
search_roi(df_labels, "AIP")

In [ ]:
AIP_ids = [
    117, 1117
]

AIP_mask, AIP_data, AIP_RDM = compute_roi_rdm(
    atlas,
    selected_volumes,
    AIP_ids,
    title="AIP RDM",
    metric="correlation",
    plot=True
)